<a href="https://colab.research.google.com/github/SinhaPushkar047/ML_Learning_Projects/blob/main/03-Heart-Disease-Risk/Heart_Disease_Risk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [250]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_curve
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score

In [251]:
pd.set_option('display.max_columns',None)

In [252]:
hr=pd.read_csv('/content/heart_disease_risk_2026.csv')

In [253]:
hr.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9000 entries, 0 to 8999
Data columns (total 27 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   patient_id                 9000 non-null   int64  
 1   age                        9000 non-null   int64  
 2   sex                        9000 non-null   object 
 3   resting_bp_systolic        9000 non-null   int64  
 4   resting_bp_diastolic       9000 non-null   int64  
 5   cholesterol_total          9000 non-null   int64  
 6   hdl                        9000 non-null   int64  
 7   ldl                        9000 non-null   int64  
 8   triglycerides              9000 non-null   int64  
 9   fasting_blood_sugar        9000 non-null   int64  
 10  hba1c                      9000 non-null   float64
 11  bmi                        9000 non-null   float64
 12  resting_heart_rate         9000 non-null   int64  
 13  max_heart_rate_achieved    9000 non-null   int64

In [254]:
cat_col=[col for col in hr.columns if hr[col].dtype=='object']

In [255]:
num_col=[col for col in hr.columns if hr[col].dtype in ['int64','float64'] and col not in ['patient_id','has_heart_disease']]

# **PreProcessing**

In [256]:
trf=ColumnTransformer([
    ('ohe',OneHotEncoder(handle_unknown='ignore',sparse_output=False),cat_col),
    ('scale',StandardScaler(),num_col)
],remainder='passthrough')

In [257]:
x_train,x_test,y_train,y_test=train_test_split(hr.drop(columns='has_heart_disease'),hr['has_heart_disease'],test_size=0.2,random_state=42,stratify=hr['has_heart_disease'])

In [258]:
x_train=trf.fit_transform(x_train)
x_test=trf.transform(x_test)

In [259]:
lr = LogisticRegression(max_iter=10000)

In [260]:
lr.fit(x_train,y_train)

LogisticRegression(max_iter=10000)

In [261]:
y_predict=lr.predict(x_test)

# **Decision Holding Tunning**

In [262]:
y_prob = lr.predict_proba(x_test)[:, 1]

In [263]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

In [264]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

J = tpr - fpr

best_index = np.argmax(J)
best_threshold = thresholds[best_index]

print(best_threshold)

0.31941040265358


In [265]:
y_pred = (y_prob >= best_threshold).astype(int)

# **Cross Val Score**

In [266]:
for scoring in ['accuracy', 'f1', 'roc_auc']:
    scores = cross_val_score(
        lr,
        x_train,y_train,
        cv=5,
        scoring=scoring
    )
    print(scoring, scores.mean())

accuracy 0.9008333333333333
f1 0.8314144416054244
roc_auc 0.9547451029005714


# **Result**

**Result Before Tunning**

In [267]:
print("\nClassification Report:\n", classification_report(y_test, y_predict))


Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.95      0.93      1255
           1       0.86      0.79      0.82       545

    accuracy                           0.90      1800
   macro avg       0.89      0.87      0.88      1800
weighted avg       0.90      0.90      0.90      1800



In [268]:
print(confusion_matrix(y_test, y_predict))

[[1186   69]
 [ 116  429]]


**Result After Tunnig**

In [269]:
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.89      0.91      1255
           1       0.78      0.85      0.81       545

    accuracy                           0.88      1800
   macro avg       0.86      0.87      0.86      1800
weighted avg       0.89      0.88      0.88      1800



In [270]:
print(confusion_matrix(y_test, y_pred))

[[1123  132]
 [  80  465]]


In [271]:
roc_auc_score(y_test, y_prob)

np.float64(0.9497905625205599)